# Test Asymmetric Masking Strategy

This notebook verifies that the improved asymmetric masking works correctly before training.

## Setup

In [ ]:
# Force reload of modules to pick up latest changes
import sys
if 'timesformer_mae' in sys.modules:
    del sys.modules['timesformer_mae']
if 'config' in sys.modules:
    del sys.modules['config']

import torch
import numpy as np
from config import get_mae_config
from timesformer_mae import TimeSformerMAE

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Create Model

In [ ]:
print("="*80)
print("TESTING ASYMMETRIC MASKING STRATEGY")
print("="*80)
print()

# Create config and model
config = get_mae_config()
model = TimeSformerMAE(config)

print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")
print()

## Test Input Generation

In [ ]:
# Create dummy input (batch_size=2, frames=7, channels=3, height=32, width=64)
B, T, C, H, W = 2, 7, 3, 32, 64
dummy_input = torch.randn(B, T, C, H, W)

print(f"Input shape: {dummy_input.shape}")
print(f"Expected patches: {T} frames × {config.num_patches_per_frame} patches/frame = {config.total_patches} total patches")
print()

## Test Patchification

In [ ]:
# Patchify
x = model.encoder.patch_embed(dummy_input)
print(f"Patchified shape: {x.shape}")
print()

## Test Masking

In [ ]:
# Apply masking
model.eval()
with torch.no_grad():
    x_masked, mask, ids_restore = model.random_masking(x)

print(f"Masked shape: {x_masked.shape}")
print(f"Mask shape: {mask.shape}")
print(f"Restore indices shape: {ids_restore.shape}")
print()

## Analyze Masking Pattern

In [ ]:
print("="*80)
print("MASKING PATTERN ANALYSIS (Sample 0)")
print("="*80)
print()

mask_np = mask[0].cpu().numpy()  # (896,)
P = config.num_patches_per_frame
mask_2d = mask_np.reshape(T, P)  # (7, 128)

# Count visible patches per frame
visible_per_frame = (1 - mask_2d).sum(axis=1)
masked_per_frame = mask_2d.sum(axis=1)

print(f"{'Frame':<8} {'Visible':<10} {'Masked':<10} {'Mask %':<10} {'Expected %':<12} {'Frame Type':<10}")
print("-" * 80)

expected_ratios = [0.90, 0.90, 0.50, 0.50, 0.50, 0.90, 0.90]
frame_types = ["BEFORE", "BEFORE", "CONTEXT", "CONTEXT", "CONTEXT", "AFTER", "AFTER"]

for t in range(T):
    vis = int(visible_per_frame[t])
    mas = int(masked_per_frame[t])
    ratio = mas / P * 100
    expected = expected_ratios[t] * 100
    frame_type = frame_types[t]

    # Check if close to expected
    status = "✓" if abs(ratio - expected) < 10 else "✗"

    print(f"{t:<8} {vis:<10} {mas:<10} {ratio:>6.1f}%    {expected:>6.1f}%       {frame_type:<10} {status}")

print()
print(f"Total visible: {visible_per_frame.sum():.0f} / {config.total_patches}")
print(f"Total masked:  {masked_per_frame.sum():.0f} / {config.total_patches}")
print(f"Overall mask ratio: {masked_per_frame.sum()/config.total_patches*100:.1f}%")
print()

## Visualize Masking Pattern

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Left: Mask pattern heatmap
im = axes[0].imshow(mask_2d, cmap='RdYlGn_r', aspect='auto', interpolation='nearest')
axes[0].set_xlabel('Patch Index (0-127)', fontsize=11)
axes[0].set_ylabel('Frame', fontsize=11)
axes[0].set_yticks(range(7))
axes[0].set_yticklabels([f'F{i}' for i in range(7)])
axes[0].set_title('Masking Pattern (Red=Masked, Green=Visible)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[0], label='Masked (1) / Visible (0)')

# Right: Bar chart
x_pos = np.arange(T)
axes[1].bar(x_pos, visible_per_frame, color='green', alpha=0.7, label='Visible')
axes[1].bar(x_pos, masked_per_frame, bottom=visible_per_frame, color='red', alpha=0.7, label='Masked')
axes[1].set_xlabel('Frame', fontsize=11)
axes[1].set_ylabel('Number of Patches', fontsize=11)
axes[1].set_title('Patches per Frame', fontsize=12, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'F{i}' for i in range(7)])
axes[1].legend()
axes[1].axhline(y=128, color='gray', linestyle='--', alpha=0.5)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Test Randomness

In [ ]:
print("="*80)
print("RANDOMNESS CHECK")
print("="*80)
print()

mask_0 = mask[0].cpu().numpy()
mask_1 = mask[1].cpu().numpy()

different = (mask_0 != mask_1).sum()
total = len(mask_0)

# Expected difference with asymmetric masking:
# Frames 0,1,5,6 (90% masked): P(different) = 2*0.9*0.1 = 0.18 → ~23 per frame × 4 = 92
# Frames 2,3,4 (50% masked): P(different) = 2*0.5*0.5 = 0.50 → ~64 per frame × 3 = 192
# Total expected: ~284 different patches (31.7%)
expected_diff_ratio = 0.30  # Expect ~30% difference due to asymmetric masking

print(f"Sample 0 vs Sample 1:")
print(f"  Different masked patches: {different} / {total} ({different/total*100:.1f}%)")
print(f"  Expected: ~{expected_diff_ratio*100:.0f}% (due to asymmetric masking)")

if different > total * 0.20:  # At least 20% should be different
    print("  ✓ Masking is properly randomized")
else:
    print("  ✗ WARNING: Masks are too similar!")

print()

## Test Forward Pass

In [ ]:
print("="*80)
print("FORWARD PASS TEST")
print("="*80)
print()

try:
    model.eval()
    with torch.no_grad():
        loss, pred, mask_out = model(dummy_input)

    print(f"✓ Forward pass successful!")
    print(f"  Loss: {loss.item():.4f}")
    print(f"  Prediction shape: {pred.shape}")
    print(f"  Mask shape: {mask_out.shape}")
    print()
    
    print("="*80)
    print("ALL TESTS PASSED! ✓")
    print("="*80)
    print()
    print("You can now run train_timesformer_mae_improved.ipynb")
    print()

except Exception as e:
    print(f"✗ Forward pass failed!")
    print(f"  Error: {str(e)}")
    print()
    raise

## Detailed Architecture Info

In [ ]:
print("="*80)
print("MODEL ARCHITECTURE")
print("="*80)
print()
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Encoder parameters: {sum(p.numel() for p in model.encoder.parameters()):,}")
print(f"Decoder parameters: {sum(p.numel() for p in model.decoder.parameters()):,}")
print()
print(f"Config:")
print(f"  Frames: {config.num_frames}")
print(f"  Image size: {config.image_height}x{config.image_width}")
print(f"  Patch size: {config.patch_size}x{config.patch_size}")
print(f"  Patches per frame: {config.num_patches_per_frame}")
print(f"  Total patches: {config.total_patches}")
print(f"  Hidden size: {config.hidden_size}")
print(f"  Encoder layers: {config.num_hidden_layers}")
print(f"  Attention heads: {config.num_attention_heads}")
print()